Create your portfolio configuration

In [1]:
import pandas as pd
import numpy as np

In [2]:
prices = pd.read_csv(
    "../data/processed/clean_prices.csv",
    index_col=0,
    parse_dates=True
)

returns = pd.read_csv(
    "../data/processed/clean_returns.csv",
    index_col=0,
    parse_dates=True
)


Defining the tickers

In [3]:
tickers = [
    "AAPL",
    "MSFT",
    "NVDA",
    "JPM",
    "XOM",
    "AMZN",
    "META",
    "GOOGL",
    "TSLA",
    "UNH"
]

In [ ]:
# Check if all tickers are present in the returns DataFrame

missing_tickers = set(tickers) - set(returns.columns)

if missing_tickers:
    raise ValueError(
        f"Missing tickers in return data: {missing_tickers}"
    )

Initial portfolio value

In [5]:
initial_portfolio_value = 1_000_000

Equal-weighted portfolio

In [6]:
n_assets = len(tickers)

equal_weights = np.ones(n_assets) / n_assets

print(equal_weights)

[0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1]


In [7]:
weights = pd.Series(
    equal_weights,
    index=tickers
)

print(weights)

AAPL     0.1
MSFT     0.1
NVDA     0.1
JPM      0.1
XOM      0.1
AMZN     0.1
META     0.1
GOOGL    0.1
TSLA     0.1
UNH      0.1
dtype: float64


Validate the weights

In [58]:
if not np.isclose(weights.sum(), 1.0):
    raise ValueError("Portfolio weights must sum to 1")

In [59]:
print("Weight sum:", weights.sum())

Weight sum: 1.0


User-defined weights

In [13]:
user_weights = {
    "AAPL": 0.20,
    "MSFT": 0.15,
    "NVDA": 0.15,
    "JPM": 0.10,
    "XOM": 0.05,
    "AMZN": 0.10,
    "META": 0.05,
    "GOOGL": 0.10,
    "TSLA": 0.05,
    "UNH": 0.05
}

In [18]:
weights = pd.Series(user_weights)
print("Weight sum:", weights.sum())

Weight sum: 1.0


Reusable weight validator

In [19]:
def validate_weights(weights, tickers):

    weights = pd.Series(weights, dtype=float)

    if set(weights.index) != set(tickers):
        raise ValueError(
            "Weights and tickers do not match"
        )

    if not np.isclose(weights.sum(), 1.0):
        raise ValueError(
            "Portfolio weights must sum to 1"
        )

    return weights

In [20]:
weights = validate_weights(
    user_weights,
    tickers
)

Calculate daily portfolio returns

This is the core calculation.

Suppose:

AAPL return = 2%
MSFT return = 1%
NVDA return = 3%

and:

AAPL weight = 40%
MSFT weight = 30%
NVDA weight = 30%

Portfolio return:

Rp = 0.4(0.02)+0.3(0.01)+0.3(0.03)
Rp=0.02

So the portfolio made: 2%

In [ ]:
# axis = 1 means we are multiplying along the columns (i.e., for each row, multiply the returns of each asset by its corresponding weight)
# .mul multiplies each stock's return by its portfolio weight.
# .sum(axis=1) sums these weighted returns across all assets for each time period, giving the total portfolio return for that period.
#  returns[tickers] selects our stocks.

portfolio_returns = returns[tickers].mul(
    weights,
    axis=1
).sum(axis=1)

print(portfolio_returns.head())

Date
2025-01-03    0.020643
2025-01-06    0.013054
2025-01-07   -0.016116
2025-01-08   -0.000831
2025-01-10   -0.010910
dtype: float64


Daily portfolio P&L

In [ ]:
# But there's an important issue here.
# We want the previous day's portfolio value, not always the original $1M.
# So we'll build the portfolio value first.

portfolio_pnl = (
    initial_portfolio_value
    * portfolio_returns
)

print(portfolio_pnl.head())

Date
2025-01-03    20642.640911
2025-01-06    13053.909010
2025-01-07   -16115.751683
2025-01-08     -831.013280
2025-01-10   -10910.458350
dtype: float64


Daily portfolio value

In [ ]:
# cumprod() calculates the cumulative product of the portfolio returns over time, which gives us the growth of the portfolio value.

portfolio_value = (
    initial_portfolio_value
    * (1 + portfolio_returns).cumprod()
)

print(portfolio_value.head())

Date
2025-01-03    1.020643e+06
2025-01-06    1.033966e+06
2025-01-07    1.017303e+06
2025-01-08    1.016457e+06
2025-01-10    1.005367e+06
dtype: float64


Calculate daily P&L correctly

In [65]:
portfolio_pnl = portfolio_value.diff()
print(portfolio_pnl.head())

Date
2025-01-03             NaN
2025-01-06    13323.376166
2025-01-07   -16663.139580
2025-01-08     -845.392201
2025-01-10   -11090.017058
dtype: float64


In [ ]:
# .iloc is used to access a group of rows and columns by integer position(s). Here, we are setting the first row (index 0) of the portfolio_pnl DataFrame to be the difference between the first value of portfolio_value and the initial_portfolio_value. This is necessary because the diff() function will return NaN for the first row since there is no previous value to subtract from.

portfolio_pnl.iloc[0] = (
    portfolio_value.iloc[0]
    - initial_portfolio_value
)

Rebalancing frequency

In [66]:
rebalancing_frequency = "monthly"

In [ ]:
# .index.to_period("M") converts the index of the returns DataFrame to a monthly period format. The .duplicated() method checks for duplicate periods, and the == False condition filters out those duplicates, leaving us with unique rebalancing dates.

rebalance_dates = returns.index[
    returns.index.to_period("M").duplicated() == False
]

Market-cap weighted portfolio

In [36]:
market_caps = {
    "AAPL": 3_000_000_000_000,
    "MSFT": 3_000_000_000_000,
    "NVDA": 4_000_000_000_000,
    "JPM": 700_000_000_000,
    "XOM": 500_000_000_000,
    "AMZN": 2_000_000_000_000,
    "META": 1_500_000_000_000,
    "GOOGL": 2_000_000_000_000,
    "TSLA": 1_000_000_000_000,
    "UNH": 500_000_000_000
}

In [37]:
market_caps = pd.Series(market_caps)

market_cap_weights = (
    market_caps / market_caps.sum()
)

In [38]:
print(market_cap_weights)
print(market_cap_weights.sum())

AAPL     0.164835
MSFT     0.164835
NVDA     0.219780
JPM      0.038462
XOM      0.027473
AMZN     0.109890
META     0.082418
GOOGL    0.109890
TSLA     0.054945
UNH      0.027473
dtype: float64
0.9999999999999999


Make portfolio construction reusable

In [39]:
def create_equal_weights(tickers):

    n = len(tickers)

    weights = np.ones(n) / n

    return pd.Series(
        weights,
        index=tickers
    )

In [40]:
weights = create_equal_weights(tickers)

print(weights)

AAPL     0.1
MSFT     0.1
NVDA     0.1
JPM      0.1
XOM      0.1
AMZN     0.1
META     0.1
GOOGL    0.1
TSLA     0.1
UNH      0.1
dtype: float64


user-defined portfolio function

In [41]:
def create_custom_weights(weight_dict, tickers):

    weights = pd.Series(
        weight_dict,
        dtype=float
    )

    weights = weights.reindex(tickers)

    if weights.isna().any():
        raise ValueError(
            "Missing weight for one or more tickers"
        )

    if not np.isclose(weights.sum(), 1.0):
        raise ValueError(
            "Weights must sum to 1"
        )

    return weights

In [68]:
weights = create_custom_weights(
    user_weights,
    tickers
)

portfolio calculation function

In [42]:
def calculate_portfolio(
    returns,
    weights,
    initial_value
):

    portfolio_returns = (
        returns
        .mul(weights, axis=1)
        .sum(axis=1)
    )

    portfolio_value = (
        initial_value
        * (1 + portfolio_returns)
        .cumprod()
    )

    portfolio_pnl = portfolio_value.diff()

    portfolio_pnl.iloc[0] = (
        portfolio_value.iloc[0]
        - initial_value
    )

    return (
        portfolio_returns,
        portfolio_pnl,
        portfolio_value
    )

In [43]:
portfolio_returns, portfolio_pnl, portfolio_value = (
    calculate_portfolio(
        returns=returns[tickers],
        weights=weights,
        initial_value=1_000_000
    )
)

final results

In [44]:
portfolio = pd.DataFrame({
    "Portfolio Return": portfolio_returns,
    "Portfolio P&L": portfolio_pnl,
    "Portfolio Value": portfolio_value
})

In [45]:
print(portfolio.head())

            Portfolio Return  Portfolio P&L  Portfolio Value
Date                                                        
2025-01-03          0.020643   20642.640911     1.020643e+06
2025-01-06          0.013054   13323.376166     1.033966e+06
2025-01-07         -0.016116  -16663.139580     1.017303e+06
2025-01-08         -0.000831    -845.392201     1.016457e+06
2025-01-10         -0.010910  -11090.017058     1.005367e+06


In [47]:
portfolio.to_csv(
    "../data/portfolio/portfolio_daily.csv"
)

In [48]:
weights.to_csv(
    "../data/portfolio/portfolio_weights.csv",
    header=["Weight"]
)

SUMMARY


import pandas as pd
import numpy as np


# ==========================================
# 1. LOAD PHASE 2 DATA
# ==========================================

prices = pd.read_csv(
    "data/clean_prices.csv",
    index_col=0,
    parse_dates=True
)

returns = pd.read_csv(
    "data/clean_returns.csv",
    index_col=0,
    parse_dates=True
)


# ==========================================
# 2. PORTFOLIO CONFIGURATION
# ==========================================

tickers = [
    "AAPL",
    "MSFT",
    "NVDA",
    "JPM",
    "XOM",
    "AMZN",
    "META",
    "GOOGL",
    "TSLA",
    "UNH"
]

initial_portfolio_value = 1_000_000

rebalancing_frequency = "monthly"


# ==========================================
# 3. EQUAL WEIGHTS
# ==========================================

n_assets = len(tickers)

weights = pd.Series(
    np.ones(n_assets) / n_assets,
    index=tickers
)


# ==========================================
# 4. VALIDATE TICKERS
# ==========================================

missing_tickers = set(tickers) - set(returns.columns)

if missing_tickers:
    raise ValueError(
        f"Missing tickers: {missing_tickers}"
    )


# ==========================================
# 5. VALIDATE WEIGHTS
# ==========================================

if not np.isclose(weights.sum(), 1.0):
    raise ValueError(
        "Portfolio weights must sum to 1"
    )


# ==========================================
# 6. CALCULATE PORTFOLIO RETURNS
# ==========================================

portfolio_returns = (
    returns[tickers]
    .mul(weights, axis=1)
    .sum(axis=1)
)


# ==========================================
# 7. CALCULATE PORTFOLIO VALUE
# ==========================================

portfolio_value = (
    initial_portfolio_value
    * (1 + portfolio_returns)
    .cumprod()
)


# ==========================================
# 8. CALCULATE DAILY P&L
# ==========================================

portfolio_pnl = portfolio_value.diff()

portfolio_pnl.iloc[0] = (
    portfolio_value.iloc[0]
    - initial_portfolio_value
)


# ==========================================
# 9. COMBINE OUTPUTS
# ==========================================

portfolio = pd.DataFrame({

    "Portfolio Return":
        portfolio_returns,

    "Portfolio P&L":
        portfolio_pnl,

    "Portfolio Value":
        portfolio_value
})


# ==========================================
# 10. DISPLAY
# ==========================================

print("\nPortfolio Weights")
print(weights)

print("\nPortfolio Results")
print(portfolio.head())

print("\nFinal Portfolio Value:")
print(portfolio_value.iloc[-1])


# ==========================================
# 11. SAVE
# ==========================================

portfolio.to_csv(
    "data/portfolio_daily.csv"
)

weights.to_csv(
    "data/portfolio_weights.csv",
    header=["Weight"]
)